# Assignment 6 — Bounding Volume Hierarchy (BVH)

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

A5's brute-force ray tracer tests every ray against every triangle. That is
*hilariously* slow — even a rabbit mesh is 10k triangles, and a Cornell box scene
at 512² × 16 spp is millions of rays.

A **BVH** wraps geometry in a tree of axis-aligned bounding boxes. To find a hit,
we test the root box, recurse into any child whose box the ray hits, and only
actually intersect triangles at the leaves. Rays skip whole subtrees the moment the
box test fails.

- **Ray/AABB** — slab method: for each axis compute `(pmin - o)/d, (pmax - o)/d`,
  and check that the interval intersections overlap.
- **SAH (Surface Area Heuristic)** — a smarter partition: pick the split that
  minimises `SA(L)*N(L) + SA(R)*N(R)`, which correlates with actual traversal cost.

See: [Bounding volume hierarchy](https://en.wikipedia.org/wiki/Bounding_volume_hierarchy), [Surface Area Heuristic (pbrt)](https://pbr-book.org/3ed-2018/Primitives_and_Intersection_Acceleration/Bounding_Volume_Hierarchies).

**The whole point of this assignment is the timing comparison** — naive vs BVH vs BVH+SAH.


In [1]:
import numpy as np

def aabb_intersect(ray_orig, inv_dir, dir_is_neg, pmin, pmax):
    t1 = (pmin - ray_orig) * inv_dir
    t2 = (pmax - ray_orig) * inv_dir
    tmin = np.minimum(t1, t2)
    tmax = np.maximum(t1, t2)
    t_enter = tmin.max()
    t_exit = tmax.min()
    return t_exit >= 0 and t_enter <= t_exit


In [2]:
def bvh_intersect(node, ray):
    inv_dir = 1.0 / ray.dir
    dir_neg = [d < 0 for d in ray.dir]
    if not aabb_intersect(ray.orig, inv_dir, dir_neg, node.pmin, node.pmax):
        return None
    if node.is_leaf:
        return node.primitive.intersect(ray)
    hL = bvh_intersect(node.left, ray)
    hR = bvh_intersect(node.right, ray)
    if hL is None: return hR
    if hR is None: return hL
    return hL if hL.t < hR.t else hR
